In [1]:
import argparse
import itertools
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [2]:
patient_id_file = pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

In [3]:
out_dict = {
    "file_path": [],
    "patient_id": [],
    "well_fov": [],
    "feature_type": [],
    "compartment": [],
    # "df_shape": [],
}

# get all well_fovs for a patient
for patient in tqdm.tqdm(patients, desc="Processing patients", leave=True):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    # print(f"Found well_fovs: {well_fovs}")
    for well_fov in tqdm.tqdm(well_fovs, desc="Processing well_fovs", leave=False):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            feature_type = feature.stem.split("_")[2]
            compartment = feature.stem.split("_")[0]
            out_dict["file_path"].append(feature)
            out_dict["patient_id"].append(patient)
            out_dict["well_fov"].append(feature.parent.stem)
            out_dict["feature_type"].append(feature_type)
            out_dict["compartment"].append(compartment)
            # out_dict["df_shape"].append(pd.read_parquet(feature).shape)
df = pd.DataFrame(out_dict)

Processing patients:   0%|          | 0/13 [00:00<?, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

...

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

In [4]:
from tqdm import tqdm

tqdm.pandas()


def safe_read_shape(x):
    try:
        df = pd.read_parquet(x)
        return df.shape, df.isna().sum().sum()
    except Exception as e:
        print(f"Error reading {x}: {e}")
        return None, None


if not pathlib.Path("../logs/feature_file_info.parquet").exists():
    df[["df_shape", "missing_values"]] = df["file_path"].progress_apply(
        lambda x: pd.Series(safe_read_shape(x))
    )
    df["file_path"] = df["file_path"].astype(str)
    df.to_parquet("../logs/feature_file_info.parquet", index=False)
else:
    df = pd.read_parquet("../logs/feature_file_info.parquet")
df

100%|██████████| 447767/447767 [1:18:29<00:00, 95.08it/s]  


,file_path,patient_id,well_fov,feature_type,compartment,df_shape,missing_values
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,SAMMed3D,Nuclei,"(15, 770)",0
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Intensity,Cytoplasm,"(15, 23)",0
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Intensity,Nuclei,"(15, 23)",0
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Organoid,"(1, 18)",0
4,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm,"(15, 18)",0
...,...,...,...,...,...,...,...
447762,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,D10-3,Intensity,Organoid,"(10, 23)",0
447763,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,D10-3,Colocalization,Organoid,"(10, 10)",0
447764,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,D10-3,Texture,Organoid,"(10, 171)",0
447765,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,D10-3,Intensity,Organoid,"(10, 23)",0


In [5]:
df = df.loc[(df["feature_type"] == "AreaSizeShape") & (df["compartment"] != "Organoid")]
df.sort_values(["patient_id", "well_fov"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

/tmp/ipykernel_4078290/149182005.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(['patient_id', 'well_fov'], inplace=True)


,file_path,patient_id,well_fov,feature_type,compartment,df_shape,missing_values
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,AreaSizeShape,Cell,"(13, 17)",2
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,AreaSizeShape,Nuclei,"(13, 17)",0
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,AreaSizeShape,Cytoplasm,"(13, 17)",0
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-2,AreaSizeShape,Cell,"(11, 17)",1
4,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-2,AreaSizeShape,Nuclei,"(11, 17)",0
...,...,...,...,...,...,...,...
5368,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-6,AreaSizeShape,Nuclei,"(13, 17)",0
5369,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-6,AreaSizeShape,Cytoplasm,"(13, 17)",0
5370,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,AreaSizeShape,Cell,"(18, 17)",0
5371,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,AreaSizeShape,Nuclei,"(18, 17)",0


In [6]:
# merge the cells, cytoplasm, and whole cell features for a given well_fov and patient_id
# check for missing values and shape of the dataframes
out_dict = {
    "patient_id": [],
    "well_fov": [],
    "path": [],
    "type": [],
}
for row in tqdm(
    df.itertuples(), total=df.shape[0], desc="Merging features", leave=True
):
    out_dict["patient_id"].append(row.patient_id)
    out_dict["well_fov"].append(row.well_fov)
    out_dict["path"].append(row.file_path)
    out_dict["type"].append(f"{row.compartment}")
out_df = pd.DataFrame(out_dict)
# pivot such that each type has its own column
out_df = out_df.pivot(
    index=["patient_id", "well_fov"], columns="type", values="path"
).reset_index()
out_df

Merging features: 100%|██████████| 5373/5373 [00:00<00:00, 568631.29it/s]


type,patient_id,well_fov,Cell,Cytoplasm,Nuclei
0,NF0014_T1,C10-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1,NF0014_T1,C10-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
2,NF0014_T1,C11-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3,NF0014_T1,C11-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
4,NF0014_T1,C2-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
...,...,...,...,...,...
1787,SARCO361_T1,G9-3,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1788,SARCO361_T1,G9-4,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1789,SARCO361_T1,G9-5,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1790,SARCO361_T1,G9-6,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...


In [13]:
labels_dict = {
    "Cell_labels": [],
    "Cytoplasm_labels": [],
    "Nuclei_labels": [],
    "patient_id": [],
    "well_fov": [],
}

# merge the dataframes and check for missing values and shape
for row in tqdm(
    out_df.itertuples(),
    total=out_df.shape[0],
    desc="Checking merged features",
    leave=True,
):
    try:
        cell_df = pd.read_parquet(row.Cell)
        cytoplasm_df = pd.read_parquet(row.Cytoplasm)
        nuclei_df = pd.read_parquet(row.Nuclei)
        labels_dict["Cell_labels"].append(cell_df["object_id"].tolist())
        labels_dict["Cytoplasm_labels"].append(cytoplasm_df["object_id"].tolist())
        labels_dict["Nuclei_labels"].append(nuclei_df["object_id"].tolist())
        labels_dict["patient_id"].append(row.patient_id)
        labels_dict["well_fov"].append(row.well_fov)
    except Exception as e:
        print(f"Error reading files for {row.patient_id} {row.well_fov}: {e}")
labels_df = pd.DataFrame(labels_dict)
labels_df

Checking merged features:   4%|▍         | 75/1792 [00:00<00:09, 185.39it/s]

Error reading files for NF0014_T1 D9-3: [Errno 2] No such file or directory: '/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/D9-3/Nuclei_NoChannel_AreaSizeShape_CPU_features.parquet'


Checking merged features:  78%|███████▊  | 1403/1792 [00:07<00:01, 215.90it/s]

Error reading files for NF0035_T1 D2-1: cannot construct a FileSource from nan
Error reading files for NF0035_T1 G9-1: cannot construct a FileSource from nan
Error reading files for NF0055_T1 F3-2: cannot construct a FileSource from nan


Checking merged features: 100%|██████████| 1792/1792 [00:09<00:00, 180.18it/s]


,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov
0,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...",NF0014_T1,C10-1
1,"[257, 514, 1028, 1285, 1542, 1799, 2056, 2313,...","[257, 514, 1028, 1285, 1542, 1799, 2056, 2313,...","[257, 514, 1028, 1285, 1542, 1799, 2056, 2313,...",NF0014_T1,C10-2
2,"[1, 3, 4, 5, 6, 7, 8]","[1, 3, 4, 5, 6, 7, 8]","[1, 3, 4, 5, 6, 7, 8]",NF0014_T1,C11-1
3,"[2, 3, 4, 5, 6, 7]","[2, 3, 4, 5, 6, 7]","[2, 3, 4, 5, 6, 7]",NF0014_T1,C11-2
4,"[257, 514, 1285, 2313, 2570, 2827]","[257, 514, 1285, 2313, 2570, 2827]","[257, 514, 1285, 2313, 2570, 2827]",NF0014_T1,C2-1
...,...,...,...,...,...
1783,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 1...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 1...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 1...",SARCO361_T1,G9-3
1784,"[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 15, 1...","[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 15, 1...","[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 15, 1...",SARCO361_T1,G9-4
1785,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...",SARCO361_T1,G9-5
1786,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]",SARCO361_T1,G9-6


In [20]:
labels_df["labels_match"] = labels_df.apply(
    lambda row: (
        (row["Cell_labels"] == row["Cytoplasm_labels"])
        and (row["Cell_labels"] == row["Nuclei_labels"])
    ),
    axis=1,
)
labels_df["same_number_of_labels"] = labels_df.apply(
    lambda row: (
        (len(row["Cell_labels"]) == len(row["Cytoplasm_labels"]))
        and (len(row["Cell_labels"]) == len(row["Nuclei_labels"]))
    ),
    axis=1,
)
labels_df["unique_labels"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df.loc[labels_df["labels_match"] == False]

,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov,labels_match,same_number_of_labels,number_of_overlapping_labels,number_of_non_overlapping_labels,unique_labels
103,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1, 2, 3, 4, 5, 7, 8, 9]",NF0014_T2,C11-5,False,False,8,8,{}
105,"[257, 514, 1285, 2313, 2827]","[257, 2313, 2827]","[257, 514, 1285, 2313, 2827]",NF0014_T2,C11-7,False,False,3,3,"{514, 1285}"
107,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[514, 771, 1028, 1285, 1542, 1799, 2056, 2313,...",NF0014_T2,C2-2,False,False,12,12,{}
132,"[257, 514, 771, 1028, 1285, 1542, 1799]","[514, 771, 1028, 1285, 1542, 1799]","[257, 514, 771, 1028, 1285, 1542, 1799]",NF0014_T2,C5-6,False,False,6,6,{257}
138,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 13, 14, 15, 16...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15...",NF0014_T2,C6-5,False,False,24,24,{11}
...,...,...,...,...,...,...,...,...,...,...
1721,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17]",SARCO361_T1,G10-4,False,False,14,14,{}
1738,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...",SARCO361_T1,G2-7,False,False,11,11,{}
1739,"[257, 514, 771, 1028, 1285, 2313, 2570, 2827, ...","[257, 514, 771, 1028, 1285, 2313, 2570, 2827, ...","[257, 771, 1028, 1285, 2313, 2570, 2827, 3084,...",SARCO361_T1,G3-1,False,False,13,13,{}
1772,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 1028, 1285, 1542, 1799, 2056, 2313]",SARCO361_T1,G7-6,False,False,8,8,{}


In [8]:
tmp_df = pd.merge(
    left=pd.merge(
        left=cell_df,
        right=cytoplasm_df,
        on=["object_id", "image_set"],
    ),
    right=nuclei_df,
    on=["object_id", "image_set"],
)
tmp_df.head()

,object_id,image_set,Cell_NoChannel_AreaSizeShape_Volume,Cell_NoChannel_AreaSizeShape_CenterX,Cell_NoChannel_AreaSizeShape_CenterY,Cell_NoChannel_AreaSizeShape_CenterZ,Cell_NoChannel_AreaSizeShape_BboxVolume,Cell_NoChannel_AreaSizeShape_MinX,Cell_NoChannel_AreaSizeShape_MaxX,Cell_NoChannel_AreaSizeShape_MinY,...,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,Nuclei_NoChannel_AreaSizeShape_MaxY,Nuclei_NoChannel_AreaSizeShape_MinZ,Nuclei_NoChannel_AreaSizeShape_MaxZ,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_SurfaceArea
0,257,G9-7,35514.0,596.456496,248.027482,4.237174,74734.0,551,637,211,...,547,636,214,290,0,7,0.626067,1,38.397933,164.473374
1,514,G9-7,95854.0,708.393526,232.935423,4.419816,185592.0,654,768,152,...,656,763,171,300,0,7,0.730017,1,51.262764,239.723780
2,771,G9-7,2148.0,58.482775,535.098231,1.132682,6480.0,47,74,507,...,36,81,484,576,1,3,0.688647,1,22.165432,19.533810
3,1028,G9-7,574253.0,871.454235,711.802387,10.713391,1209775.0,761,984,630,...,813,962,674,808,1,16,0.476510,1,64.836355,666.873509
4,1285,G9-7,323827.0,1009.518073,1114.801866,8.097759,675990.0,925,1099,1022,...,960,1039,1088,1185,1,16,0.478551,1,47.185270,383.419902
